<a href="https://colab.research.google.com/github/carolmagalhaees/arraial-beira-rio/blob/main/musiclearn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MusicLearn - Acelerador Colab

Fornece GPU para **Whisper** (transcrição) e **Ollama** (análise de palavras).

1. Execute as células em ordem
2. Copie o URL do ngrok gerado
3. Coloque no `OLLAMA_HOST` do seu `.env` local

**Importante:** Mantenha esta aba aberta enquanto usar o app.

In [1]:
# 1. Verificar GPU
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('AVISO: sem GPU ativa — vá em Ambiente de execução > Alterar tipo de runtime > T4 GPU')

CUDA disponível: True
GPU: Tesla T4


In [2]:
# 2. Instalar dependências
!apt-get update -qq
!apt-get install -y -qq curl ffmpeg 2>/dev/null | tail -1

# Whisper
!pip install -q openai-whisper

# ngrok
!pip install -q pyngrok
print('Dependências instaladas')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 48.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependências instaladas


Precisamos instalar o pacote `zstd` para que o Ollama possa ser extraído corretamente.

In [8]:
!apt-get install -y -qq zstd

Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [9]:
# 3. Instalar Ollama via script oficial
import subprocess, time

print('Instalando Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh
print('Instalação concluída')

Instalando Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Instalação concluída


In [10]:
# 4. Iniciar Ollama server em background
import os, subprocess
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Mata processo existente se houver
subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(1)

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print(f'Ollama rodando (PID: {ollama_proc.pid})')

Ollama rodando (PID: 3181)


In [7]:
!curl -fsSL https://ollama.com/install.sh | sudo sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [11]:
# 5. Baixar modelo llama3.2:3b (otimizado para GPU)
import subprocess

print('Baixando llama3.2:3b...')
result = subprocess.run(
    ['ollama', 'pull', 'llama3.2:3b'], # Removido OLLAMA_DIR pois ollama está agora no PATH
    capture_output=True, text=True
)
if result.returncode == 0:
    print('Modelo baixado com sucesso!')
else:
    print('Erro:', result.stderr)

Baixando llama3.2:3b...
Modelo baixado com sucesso!


In [14]:
# 6. Testar Ollama com uma palavra
import requests, json

# Define o payload como um dicionário Python
payload = {
  "model": "llama3.2:3b",
  "prompt": "Analise a palavra 'lovely' no contexto: 'Isn't it lovely, all alone'.\nRetorne APENAS um JSON:\n{\"type\": \"...\", \"definition\": \"...\", \"contextNote\": \"...\", \"expression\": \"...\", \"phonetic\": \"...\", \"synonyms\": [...], \"exampleEn\": \"...\", \"examplePt\": \"...\", \"difficulty\": \"...\"}",
  "format": "json",
  "stream": False
}

try:
    # Aumentando o timeout para dar mais tempo ao modelo carregar
    r = requests.post('http://localhost:11434/api/generate', json=payload, timeout=120)
    print('Resposta:', r.json()['response'][:200])
except Exception as e:
    print('Erro no teste:', e)

Erro no teste: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=30)


In [24]:
# 7. Expor com ngrok
from pyngrok import ngrok

ngrok.set_auth_token('3GWDSXjoWfCwu5AnrASiMXvhBK6_TpY5xgFLwzbNQzzQ4MfD')

tunnel = ngrok.connect(11434, 'http')
public_url = tunnel.public_url
print('OLLAMA_HOST =', public_url)

OLLAMA_HOST = https://edge-expulsion-sweat.ngrok-free.dev


In [20]:
# 8. (Opcional) Servidor proxy para Whisper
# Se também quiser usar Whisper com GPU do Colab

from flask import Flask, request, jsonify
import whisper, threading

whisper_app = Flask(__name__)
whisper_model = None

@whisper_app.route('/whisper/transcribe', methods=['POST'])
def transcribe():
    global whisper_model
    if whisper_model is None:
        whisper_model = whisper.load_model('small')
    data = request.json
    audio_path = data.get('audio_path', '')
    if not audio_path:
        return jsonify({'error': 'audio_path required'}), 400
    result = whisper_model.transcribe(audio_path, language='en', word_timestamps=True)
    segments = [{'start': s['start'], 'end': s['end'], 'text': s['text']} for s in result['segments']]
    return jsonify({'segments': segments})

def run_whisper_server():
    whisper_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=run_whisper_server, daemon=True)
t.start()

# Tunnel para Whisper
tunnel2 = ngrok.connect(5000, 'http')
print('URL do Whisper (se for usar):')
print(f'  {tunnel2.public_url}')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


URL do Whisper (se for usar):
  https://edge-expulsion-sweat.ngrok-free.dev


In [23]:
# 9. Manter vivo
print('\nTudo pronto!')
print(f'OLLAMA_HOST = {public_url}')
print('Copie esta URL para o arquivo .env do backend local.')
print('\nNÃO feche esta aba — enquanto estiver aberta, o serviço funciona.')


Tudo pronto!
OLLAMA_HOST = https://edge-expulsion-sweat.ngrok-free.dev
Copie esta URL para o arquivo .env do backend local.

NÃO feche esta aba — enquanto estiver aberta, o serviço funciona.
